In [66]:
import pandas as pd

df_crimes = pd.read_csv(
    "./output/maceio/maceio_with_grid_index.csv",
    delimiter=";",
    parse_dates=["DATA_HORA"]
)

In [67]:
display(df_crimes)
df_crimes.info()
years = sorted(df_crimes['DATA_HORA'].dt.year.unique())
display(years)

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell
0,0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió,825
1,1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió,863
2,2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió,903
3,3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió,902
4,4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió,863
...,...,...,...,...,...,...,...
72707,72707,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió,1175
72708,72708,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió,903
72709,72709,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió,1184
72710,72710,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió,1063


<class 'pandas.DataFrame'>
RangeIndex: 72712 entries, 0 to 72711
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Unnamed: 0   72712 non-null  int64         
 1   LONGITUDE    72712 non-null  float64       
 2   LATITUDE     72712 non-null  float64       
 3   ROUBO_GROUP  72712 non-null  str           
 4   DATA_HORA    72712 non-null  datetime64[us]
 5   CIDADE_FATO  72712 non-null  str           
 6   cell         72712 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 3.9 MB


[np.int32(2012),
 np.int32(2013),
 np.int32(2014),
 np.int32(2015),
 np.int32(2016),
 np.int32(2017),
 np.int32(2018),
 np.int32(2019),
 np.int32(2020),
 np.int32(2021),
 np.int32(2022)]

In [68]:
crimes_types = sorted(df_crimes['ROUBO_GROUP'].unique())
display(crimes_types)

['commercial_robbery',
 'public_transport_robbery',
 'residential_robbery',
 'street_robbery',
 'vehicle_robbery']

In [69]:
import numpy as np
import datetime
from pandarallel import pandarallel
import os

cpu_count = os.cpu_count() or 1
nb_workers = max(1, min(cpu_count, 24))
pandarallel.initialize(nb_workers=nb_workers, progress_bar=True)


def hour_of_year(dt):
    beginning_of_year = datetime.datetime(
        dt["DATA_HORA"].year, 1, 1, tzinfo=dt["DATA_HORA"].tzinfo
    )
    return pd.Series(
        {
            "hour_year": (dt["DATA_HORA"] - beginning_of_year).total_seconds()
            // 3600
        }
    )


df_crimes
display("Translate date of crime to hour of the year")
df_hour = df_crimes.merge(
    df_crimes.parallel_apply(hour_of_year, axis=1), left_index=True, right_index=True
)
display("Initial shape: ", df_hour.shape)
display("Initial shape: ", df_hour)

INFO: Pandarallel will run on 24 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


'Translate date of crime to hour of the year'

'Initial shape: '

(72712, 8)

'Initial shape: '

,Unnamed: 0,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO,cell,hour_year
0,0,-35.698388,-9.658217,street_robbery,2022-01-01 02:00:00,Maceió,825,2.0
1,1,-35.703405,-9.660662,street_robbery,2022-01-01 02:00:00,Maceió,863,2.0
2,2,-35.710142,-9.653007,vehicle_robbery,2022-01-01 02:00:00,Maceió,903,2.0
3,3,-35.713147,-9.668955,street_robbery,2022-01-01 03:00:00,Maceió,902,3.0
4,4,-35.703983,-9.663635,street_robbery,2022-01-01 04:00:00,Maceió,863,4.0
...,...,...,...,...,...,...,...,...
72707,72707,-35.756315,-9.662037,street_robbery,2014-12-22 12:50:00,Maceió,1175,8532.0
72708,72708,-35.715706,-9.652401,vehicle_robbery,2014-12-22 19:00:00,Maceió,903,8539.0
72709,72709,-35.755908,-9.586733,residential_robbery,2014-12-23 05:30:00,Maceió,1184,8549.0
72710,72710,-35.738317,-9.624963,street_robbery,2014-12-23 09:30:00,Maceió,1063,8553.0


In [ ]:
GRID_SIZE = 39

for ct in crimes_types:
    df_per_crime = df_hour[df_hour["ROUBO_GROUP"] == ct].copy()
    df_all_years_list = []
    for y in years:
        dfy = df_per_crime[df_per_crime["DATA_HORA"].dt.year == y].copy()

        # count crimes by (cell, DATA_HORA)
        counts = dfy.groupby(["cell", "hour_year"]).size()
        display(counts)

        grid = counts.unstack("hour_year") 


        beginning_of_y1 = datetime.datetime(y, 1, 1)
        beginning_of_y2 = datetime.datetime(y+1, 1, 1)
        num_hours_total = int((beginning_of_y2 - beginning_of_y1).total_seconds() // 3600)
        grid = grid.reindex(columns=range(num_hours_total))
        grid = grid.reindex(index=range(GRID_SIZE**2))
        df_year = grid.copy()
        df_year.columns = list(map(lambda x: str(x) + f"_{str(y)}", df_year.columns.tolist()))
        df_all_years_list.append(df_year)

  
    df_all = pd.concat(df_all_years_list,axis=1)
    df_all.T.to_csv(f"./output/maceio/maceio_{ct}_all_grid.csv")


cell  hour_year
370   6891.0       1
722   963.0        1
      1443.0       1
      4035.0       1
      4923.0       1
                  ..
1307  1587.0       1
      1971.0       1
      6171.0       1
1384  579.0        1
      4059.0       1
Length: 365, dtype: int64

cell  hour_year
103   5857.0       1
180   6964.0       1
255   3984.0       1
      4297.0       1
524   7493.0       1
                  ..
1346  5395.0       1
      7285.0       1
1381  6781.0       1
      7910.0       1
1418  1054.0       1
Length: 808, dtype: int64

cell  hour_year
218   6865.0       1
523   825.0        1
561   107.0        1
      8414.0       1
599   6429.0       1
                  ..
1385  1096.0       1
1418  1717.0       1
1423  2175.0       1
1424  1984.0       1
      4118.0       1
Length: 601, dtype: int64

cell  hour_year
179   973.0        1
293   3265.0       1
      8737.0       1
561   6927.0       1
599   7099.0       1
                  ..
1381  4034.0       1
      5736.0       1
1384  5106.0       1
1418  522.0        1
1424  6625.0       1
Length: 310, dtype: int64

cell  hour_year
103   4316.0       1
294   117.0        1
524   5611.0       1
562   2830.0       1
      2974.0       1
                  ..
1384  6228.0       1
      8566.0       1
1385  4461.0       1
      8708.0       1
1497  7837.0       1
Length: 334, dtype: int64

...

cell  hour_year
293   96.0         1
      7463.0       1
      7991.0       1
294   118.0        1
      4793.0       1
                  ..
1424  7605.0       1
      8084.0       1
1459  3777.0       1
      6575.0       1
1498  7845.0       1
Length: 936, dtype: int64

cell  hour_year
179   6578.0       1
293   4795.0       1
294   1777.0       1
370   5376.0       1
447   2374.0       1
                  ..
1424  2746.0       1
      3810.0       1
      8331.0       1
1498  4094.0       1
      4317.0       1
Length: 538, dtype: int64

cell  hour_year
370   7390.0       1
524   1156.0       1
      7796.0       1
562   7560.0       1
      7961.0       1
                  ..
1458  2160.0       1
1459  697.0        1
      2036.0       1
1497  919.0        1
      5464.0       1
Length: 456, dtype: int64

cell  hour_year
103   3890.0       1
179   7753.0       1
372   1188.0       1
409   4345.0       1
523   3250.0       1
                  ..
1497  5040.0       1
      8736.0       1
1498  2585.0       1
      2836.0       1
      5914.0       1
Length: 343, dtype: int64

cell  hour_year
25    5597.0       1
103   1770.0       1
141   2276.0       1
217   496.0        1
255   356.0        1
                  ..
1424  1700.0       1
      2418.0       1
      7645.0       1
1497  2901.0       1
1498  8490.0       1
Length: 500, dtype: int64